
# 05 | Council Data Cleaning — Silver Layer

**Project:** Victorian Local Government Financial Sustainability & Performance Intelligence (2020–2025)

## Objective
Transform Bronze performance data into a validated
Silver Delta table for financial and service reporting.

## Inputs
- workspace.default.bronze_performance
- Council category reference CSV

## Output
- workspace.default.silver_performance

## Business requirements
1. Preserve all 23,068 source records.
2. Standardise council names to 79 identities.
3. Reconcile council groups into five categories.
4. Convert Result and Target to numeric types.
5. Preserve original source values for auditing.
6. Validate data quality before saving Silver.

**Grain:** Council × FinancialYear × Indicator ID

Missing values remain null, not zero.


In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.default.bronze_performance"
SILVER = "workspace.default.silver_performance"
REFERENCE_PATH = "/Volumes/workspace/default/council_raw/Council category data.csv"
EXPECTED_YEARS = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

bronze = spark.table(BRONZE)
required = {"Council", "Group", "FinancialYear", "ID", "Result", "Target"}
assert required.issubset(set(bronze.columns)), f"Missing columns: {required - set(bronze.columns)}"
bronze_rows = bronze.count()
print("Bronze rows:", bronze_rows)
print("Bronze columns:", bronze.columns)
display(bronze.groupBy("FinancialYear").count().orderBy("FinancialYear"))

Bronze rows: 23068
Bronze columns: ['Council', 'Group', 'ID', 'Description', 'Service_Provided', 'Data_Applicable', 'Result', 'Variation', 'FinancialYear', 'Target']


FinancialYear,count
2020-21,4582
2021-22,4582
2022-23,4582
2023-24,4661
2024-25,4661



## Step 1 — Standardise Council Names

Notebook 04 identified 81 original council-name strings
representing 79 council identities.

Apply the three validated mappings while preserving
the original source names for auditability.


In [0]:

clean = (
    bronze
    .withColumn("OriginalCouncil", F.col("Council"))
    .withColumn("OriginalGroup", F.col("Group"))
    .withColumn("OriginalResult", F.col("Result"))
    .withColumn("OriginalTarget", F.col("Target"))
)

name = F.trim(F.col("OriginalCouncil"))

clean = clean.withColumn(
    "Council",
    F.when(name == "Moreland City", "Merri-bek City")
     .when(name == "Colac-Otway Shire", "Colac Otway Shire")
     .when(
         name == "Greater Shepparton",
         "Greater Shepparton City"
     )
     .otherwise(name)
)

print(
    "Raw council names:",
    clean.select("OriginalCouncil").distinct().count()
)

print(
    "Standardised councils:",
    clean.select("Council").distinct().count()
)

display(
    clean.filter(
        F.col("OriginalCouncil") != F.col("Council")
    )
    .select("OriginalCouncil", "Council")
    .distinct()
    .orderBy("OriginalCouncil")
)


Raw council names: 81
Standardised councils: 79


OriginalCouncil,Council
Colac-Otway Shire,Colac Otway Shire
Greater Shepparton,Greater Shepparton City
Moreland City,Merri-bek City


## Step 2 — Standardise Council Categories

Convert the historical group-label variants into:

1. Metropolitan
2. Interface
3. Regional
4. Large Shire
5. Small Shire

Reconcile the standardised categories against the
council reference dataset.

Do not automatically overwrite historical
classifications without validation.



In [0]:

def council_key(c):
    return F.regexp_replace(
        F.lower(F.trim(c)),
        r"\s+council$", ""
    )

def category_label(c):
    x = F.upper(
        F.regexp_replace(F.trim(c), "_", " ")
    )

    return (
        F.when(
            x.isin("METRO", "METROPOLITAN"),
            "Metropolitan"
        )
        .when(x == "INTERFACE", "Interface")
        .when(
            x.isin("REGIONAL", "REGIONAL CITY"),
            "Regional"
        )
        .when(x == "LARGE SHIRE", "Large Shire")
        .when(x == "SMALL SHIRE", "Small Shire")
        .otherwise(F.lit(None).cast("string"))
    )

clean = clean.withColumn(
    "Group",
    category_label(F.col("OriginalGroup"))
)

categories = (
    spark.read
    .option("header", True)
    .csv(REFERENCE_PATH)
)

assert {
    "Council", "Council Category"
}.issubset(set(categories.columns))

reference = (
    categories
    .withColumn(
        "Council_Key",
        council_key(F.col("Council"))
    )
    .withColumn(
        "Reference_Group",
        category_label(F.col("Council Category"))
    )
    .select("Council_Key", "Reference_Group")
)

print("Reference rows:", reference.count())

display(
    clean.groupBy("Group")
    .count()
    .orderBy("Group")
)


Reference rows: 79


Group,count
Interface,2628
Large Shire,5548
Metropolitan,6424
Regional,2920
Small Shire,5548



## Step 3 — Convert Reporting Measures

Convert Result and Target into numeric values.

Preserve:
- OriginalResult
- OriginalTarget

Treat missing markers as null.

Do not replace missing targets with zero because
zero and missing have different business meanings.

Unexpected nonnumeric values must be investigated.


In [0]:

missing_tokens = [
    "", "null", "n/a", "na",
    "-", "not available"
]

for column in ["Result", "Target"]:

    raw = F.trim(
        F.col("Original" + column).cast("string")
    )

    clean = clean.withColumn(
        column,
        F.when(
            raw.isNull() |
            F.lower(raw).isin(*missing_tokens),
            F.lit(None).cast("double")
        )
        .otherwise(
            F.expr(
                f"try_cast(trim(`Original{column}`) AS DOUBLE)"
            )
        )
    )

clean.select(
    F.count("*").alias("Rows"),
    F.count("Result").alias("Numeric_Results"),
    F.count("Target").alias("Numeric_Targets"),
    F.sum(
        F.col("Result").isNull().cast("int")
    ).alias("Null_Results"),
    F.sum(
        F.col("Target").isNull().cast("int")
    ).alias("Null_Targets")
).show()

for column in ["Result", "Target"]:

    raw = F.trim(
        F.col("Original" + column).cast("string")
    )

    invalid = clean.filter(
        raw.isNotNull() &
        ~F.lower(raw).isin(*missing_tokens) &
        F.col(column).isNull()
    )

    print(
        f"Unexpected nonnumeric {column} values:",
        invalid.count()
    )

    display(
        invalid.select(
            "Council",
            "FinancialYear",
            "ID",
            "Original" + column
        ).limit(10)
    )


+-----+---------------+---------------+------------+------------+
| Rows|Numeric_Results|Numeric_Targets|Null_Results|Null_Targets|
+-----+---------------+---------------+------------+------------+
|23068|          22926|           1433|         142|       21635|
+-----+---------------+---------------+------------+------------+

Unexpected nonnumeric Result values: 0


Council,FinancialYear,ID,OriginalResult


Unexpected nonnumeric Target values: 0


Council,FinancialYear,ID,OriginalTarget



## Step 4 — Silver Data Quality Validation

Validate:
- Source record count preservation
- 79 standardised councils
- Five financial years
- Five council categories
- No duplicate business keys
- No unmatched council identities
- No duplicate reference keys
- No category mismatches
- No unexpected numeric conversion failures

The Silver table must not be saved if a quality gate fails.


In [0]:

council_count = clean.select(
    "Council"
).distinct().count()

years = sorted(
    r[0]
    for r in clean.select(
        "FinancialYear"
    ).distinct().collect()
)

group_count = clean.select(
    "Group"
).distinct().count()

business_key_duplicates = (
    clean.groupBy(
        "Council", "FinancialYear", "ID"
    )
    .count()
    .filter(F.col("count") > 1)
)

source_keys = clean.select(
    council_key(
        F.col("Council")
    ).alias("Council_Key")
).distinct()

reference_keys = reference.select(
    "Council_Key"
).distinct()

unmatched_source = source_keys.join(
    reference_keys,
    "Council_Key",
    "left_anti"
)

unmatched_reference = reference_keys.join(
    source_keys,
    "Council_Key",
    "left_anti"
)

reference_duplicates = (
    reference.groupBy("Council_Key")
    .count()
    .filter(F.col("count") > 1)
)

comparison = (
    clean.withColumn(
        "Council_Key",
        council_key(F.col("Council"))
    )
    .join(reference, "Council_Key", "left")
)

category_mismatches = comparison.filter(
    F.col("Group").isNull() |
    F.col("Reference_Group").isNull() |
    (
        F.col("Group") !=
        F.col("Reference_Group")
    )
)

unexpected_numeric = {}

for column in ["Result", "Target"]:

    raw = F.trim(
        F.col("Original" + column).cast("string")
    )

    unexpected_numeric[column] = (
        clean.filter(
            raw.isNotNull() &
            ~F.lower(raw).isin(*missing_tokens) &
            F.col(column).isNull()
        ).count()
    )

checks = {
    "Row count preserved":
        clean.count() == bronze_rows == 23068,

    "79 councils":
        council_count == 79,

    "Five expected financial years":
        years == EXPECTED_YEARS,

    "Five peer categories":
        group_count == 5,

    "No duplicate business keys":
        business_key_duplicates.count() == 0,

    "No unmatched source names":
        unmatched_source.count() == 0,

    "No unmatched reference names":
        unmatched_reference.count() == 0,

    "No duplicate reference keys":
        reference_duplicates.count() == 0,

    "No category mismatches":
        category_mismatches.count() == 0,

    "No unexpected Result conversion failures":
        unexpected_numeric["Result"] == 0,

    "No unexpected Target conversion failures":
        unexpected_numeric["Target"] == 0,
}

for label, passed in checks.items():
    print(
        ("PASS" if passed else "FAIL")
        + " | " + label
    )

if category_mismatches.count() > 0:
    display(
        category_mismatches.select(
            "Council",
            "FinancialYear",
            "OriginalGroup",
            "Group",
            "Reference_Group"
        ).distinct().limit(30)
    )

if business_key_duplicates.count() > 0:
    display(
        business_key_duplicates.limit(30)
    )

assert all(checks.values()), (
    "Silver validation failed. "
    "Review the failed checks before saving."
)

print(
    "All quality gates passed. Ready to save Silver."
)


PASS | Row count preserved
PASS | 79 councils
PASS | Five expected financial years
PASS | Five peer categories
PASS | No duplicate business keys
PASS | No unmatched source names
PASS | No unmatched reference names
PASS | No duplicate reference keys
PASS | No category mismatches
PASS | No unexpected Result conversion failures
PASS | No unexpected Target conversion failures
All quality gates passed. Ready to save Silver.



## Step 5 — Save Silver Delta Table

Save the cleaned and validated performance dataset as:

workspace.default.silver_performance

Only execute this step after all quality gates pass.

The existing Silver table, if present, will be
overwritten with the validated output.


In [0]:

(
    clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER)
)

saved = spark.table(SILVER)

assert saved.count() == bronze_rows, (
    "Read-back row count does not match Bronze"
)

assert (
    saved.select("Council").distinct().count()
    == 79
)

print("Saved and verified:", SILVER)
print("Rows:", saved.count())

saved.printSchema()

display(
    saved.select(
        "Council",
        "Group",
        "FinancialYear",
        "ID",
        "Result",
        "Target",
        "OriginalCouncil",
        "OriginalGroup"
    ).limit(10)
)


Saved and verified: workspace.default.silver_performance
Rows: 23068
root
 |-- Council: string (nullable = true)
 |-- Group: string (nullable = true)
 |-- ID: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Service_Provided: string (nullable = true)
 |-- Data_Applicable: string (nullable = true)
 |-- Result: double (nullable = true)
 |-- Variation: string (nullable = true)
 |-- FinancialYear: string (nullable = true)
 |-- Target: double (nullable = true)
 |-- OriginalCouncil: string (nullable = true)
 |-- OriginalGroup: string (nullable = true)
 |-- OriginalResult: string (nullable = true)
 |-- OriginalTarget: string (nullable = true)



Council,Group,FinancialYear,ID,Result,Target,OriginalCouncil,OriginalGroup
Southern Grampians Shire,Large Shire,2020-21,AM1,1.205349439,null,Southern Grampians Shire,LARGE_SHIRE
South Gippsland Shire,Large Shire,2020-21,AM1,2.04040404,null,South Gippsland Shire,LARGE_SHIRE
Stonnington City,Metropolitan,2020-21,AM1,1.80608365,null,Stonnington City,METRO
Ararat Rural City,Small Shire,2020-21,AM1,1.0,null,Ararat Rural City,SMALL_SHIRE
Strathbogie Shire,Small Shire,2020-21,AM1,1.69982238,null,Strathbogie Shire,SMALL_SHIRE
Surf Coast Shire,Large Shire,2020-21,AM1,1.350993377,null,Surf Coast Shire,LARGE_SHIRE
Swan Hill Rural City,Large Shire,2020-21,AM1,1.074164629,null,Swan Hill Rural City,LARGE_SHIRE
Towong Shire,Small Shire,2020-21,AM1,1.945945946,null,Towong Shire,SMALL_SHIRE
Wellington Shire,Large Shire,2020-21,AM1,1.17572892,null,Wellington Shire,LARGE_SHIRE
West Wimmera Shire,Small Shire,2020-21,AM1,1.0,null,West Wimmera Shire,SMALL_SHIRE




## Results and Outcomes

### Dataset
- Source: workspace.default.bronze_performance
- Output: workspace.default.silver_performance
- Records preserved: 23,068
- Silver columns: 14
- Financial years: 5

### Council Standardisation
- Original council-name strings: 81
- Standardised council identities: 79
- Standardised council categories: 5
- Unmatched council names: 0
- Category mismatches: 0

### Numeric Conversion
- Valid numeric Results: 22,926
- Missing Results: 142
- Valid numeric Targets: 1,433
- Missing Targets: 21,635
- Unexpected conversion failures: 0

### Data Quality and Assurance
- Duplicate business keys: 0
- Duplicate reference keys: 0
- Quality checks passed: 11/11
- Silver table successfully saved and read back

### Business Outcome
Created a consistent, auditable Silver performance
dataset covering 79 Victorian councils across five
financial years.

The dataset supports historical service-performance
analysis and peer benchmarking in Power BI.

### Reporting Limitations
- Missing targets remain null.
- Council categories are reconciled against the
  supplied reference dataset.
- Financial sustainability and per-capita KPIs
  require additional financial and population data.

### Next Step
Notebook 06 — Financial and Population Data Integration.
